# Tugas 1 - Klasifikasi Wine Quality dengan KNN
  
| Nama              | NRP        |
|-------------------|------------|
|Muhamad Nafi Mulyo              |5054251005       |

## Deskripsi Tugas
Pada tugas ini, kita akan menggunakan Wine Quality Dataset. Dataset bisa diakses melalui link berikut:\
🔗 https://www.kaggle.com/datasets/yasserh/wine-quality-dataset

Tujuan utama dari tugas ini adalah membangun model K-Nearest Neighbors (KNN) untuk mengklasifikasikan kualitas wine. 

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan Dataset & Eksplorasi Awal

- Memuat dataset, melihat struktur data, dan distribusi label.

2. Preprocessing 
- Memproses data agar siap untuk digunakan dalam membangun model.

3. Eksperimen Model KNN
- Bangun model KNN dengan mencoba beberapa nilai k (misalnya 3, 5, dan 7 --> BEBAS) serta dua metric jarak (seperti Euclidean dan Manhattan).
- Eksperimen ini bertujuan untuk membandingkan performa KNN dengan parameter yang berbeda.

4. Evaluasi Model
- Hitung metrik evaluasi seperti Accuracy, Precision, Recall, F1-Score, serta visualisasikan Confusion Matrix.

5. Analisis & Kesimpulan
- Bandingkan hasil antar eksperimen yang telah dilakukan dan berikan kesimpulan.


# 1. Persiapan Dataset & Eksplorasi Awal

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
import os
# Load dataset (fleksibel baik dibuka dari root monorepo atau dari folder tugas)
data_path = "archive/WineQT.csv"
if not os.path.exists(data_path) and os.path.exists("machine-learning/tugas-1-knn-wine-quality/archive/WineQT.csv"):
    data_path = "machine-learning/tugas-1-knn-wine-quality/archive/WineQT.csv"

df = pd.read_csv(data_path)

# Drop kolom Id jika ada
if "Id" in df.columns:
    df = df.drop(columns=["Id"])

df.head()

In [ ]:
# Informasi ringkas dataset
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Distribusi label quality
plt.figure(figsize=(7, 5))
sns.countplot(x='quality', data=df, hue='quality', legend=False, palette='viridis')
plt.title('Distribusi Quality Wine')
plt.xlabel('Quality')
plt.ylabel('Jumlah')
plt.show()

## Exploratory Data Analysis (EDA) Tambahan

In [ ]:
# Hubungan antara fitur utama (alcohol & volatile acidity) dengan quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='quality', y='alcohol', data=df, hue='quality', legend=False, ax=axes[0], palette='Set2')
axes[0].set_title('Hubungan Alcohol vs Quality')

sns.boxplot(x='quality', y='volatile acidity', data=df, hue='quality', legend=False, ax=axes[1], palette='Set2')
axes[1].set_title('Hubungan Volatile Acidity vs Quality')

plt.tight_layout()
plt.show()

# 2. Preprocessing

In [ ]:
# Cek missing values dan data duplikat
print("Missing values per kolom:")
print(df.isnull().sum())
print("
Jumlah data duplikat:", df.duplicated().sum())

# Hapus data duplikat jika ditemukan
if df.duplicated().sum() > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("
Data duplikat berhasil dihapus. Ukuran data baru:", df.shape)

In [ ]:
# Inspeksi persebaran outliers pada fitur-fitur numerik
plt.figure(figsize=(14, 6))
sns.boxplot(data=df.drop(columns=['quality']), palette='Set3')
plt.xticks(rotation=45)
plt.title('Inspeksi Outliers pada Fitur Wine')
plt.tight_layout()
plt.show()

In [ ]:
# Pemisahan fitur (X) dan label target (y)
X = df.drop(columns=['quality'])
y = df['quality']

print("Ukuran X (Fitur):", X.shape)
print("Ukuran y (Target):", y.shape)

# 3. Eksperimen Model KNN


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Train-Test Split (80% train, 20% test) dengan stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Ukuran Data Train:", X_train_scaled.shape)
print("Ukuran Data Test:", X_test_scaled.shape)

In [ ]:
# Eksperimen variasi nilai K dan metric jarak (Euclidean & Manhattan)
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
metrics = ['euclidean', 'manhattan']

results = []

for metric in metrics:
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
        knn.fit(X_train_scaled, y_train)
        
        train_acc = knn.score(X_train_scaled, y_train)
        test_acc = knn.score(X_test_scaled, y_test)
        
        results.append({
            'Metric': metric.capitalize(),
            'K': k,
            'Train Accuracy': round(train_acc, 4),
            'Test Accuracy': round(test_acc, 4)
        })

results_df = pd.DataFrame(results)
results_df

# 4. Evaluasi Model


In [ ]:
# Visualisasi performa akurasi vs nilai K
plt.figure(figsize=(10, 5))
sns.lineplot(data=results_df, x='K', y='Test Accuracy', hue='Metric', marker='o', linewidth=2)
plt.title('Perbandingan Akurasi Test vs Nilai K (Euclidean vs Manhattan)')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi Test')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Evaluasi model terbaik (K=9 dengan Manhattan)
best_k = 9
best_metric = 'manhattan'

best_knn = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric)
best_knn.fit(X_train_scaled, y_train)

y_pred = best_knn.predict(X_test_scaled)

print(f"--- Evaluasi Model Terbaik (K={best_k}, Metric={best_metric.capitalize()}) ---")
print("Accuracy Score:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.title(f'Confusion Matrix (KNN K={best_k}, {best_metric.capitalize()})')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# 5. Analisis


Berdasarkan serangkaian eksperimen yang telah dilakukan pada model **K-Nearest Neighbors (KNN)** untuk mengklasifikasikan kualitas wine, berikut adalah analisis mendalam yang didapatkan:

1. **Pengaruh Nilai K terhadap Overfitting dan Generalisasi**:
   - Pada $K=1$, akurasi pada data latih (*Train Accuracy*) mencapai 100%, namun akurasi pada data uji (*Test Accuracy*) relatif lebih rendah. Hal ini menunjukkan bahwa nilai $K$ yang sangat kecil menyebabkan model **overfitting** (sensitif terhadap noise pada data latih).
   - Seiring bertambahnya nilai $K$ (seperti $K=7$ atau $K=9$), perbedaan antara akurasi train dan test semakin mengecil, menunjukkan bahwa model semakin mampu melakukan **generalisasi** pada data baru.

2. **Perbandingan Metric Jarak (Euclidean vs Manhattan)**:
   - Jarak **Manhattan** ($p=1$) memberikan performa akurasi yang sedikit lebih baik dan stabil dibanding jarak **Euclidean** ($p=2$) pada sebagian besar nilai $K$.
   - Hal ini disebabkan karena dataset Wine Quality memiliki dimensi fitur numerik yang cukup banyak (11 fitur), di mana jarak Manhattan cenderung lebih tahan (*robust*) terhadap data multidimensi.

3. **Pentingnya Feature Scaling (StandardScaler)**:
   - Karena KNN bergantung pada perhitungan jarak antar sampel, fitur dengan rentang angka besar (seperti `total sulfur dioxide`) akan mendominasi fitur berentang kecil (seperti `volatile acidity`) jika tidak di-scale. Penggunaan `StandardScaler` terbukti membuat kontribusi setiap fitur seimbang.

4. **Tantangan Class Imbalance**:
   - Berdasarkan evaluasi confusion matrix dan classification report, model berkinerja sangat baik pada kelas kualitas mayoritas (`5` dan `6`), namun kesulitan pada kelas ekstrem/minoritas (`3`, `4`, dan `8`).

# 6. Kesimpulan dan Saran


### Kesimpulan
1. Model KNN berhasil dibangun untuk mengklasifikasikan kualitas wine berdasarkan sifat fisika & kimianya.
2. Kombinasi parameter terbaik yang diperoleh dari eksperimen adalah **$K = 9$** dengan **Manhattan Distance**, yang menghasilkan kestabilan akurasi data uji terbaik tanpa overfitting berlebihan.
3. *Feature scaling* menggunakan `StandardScaler` merupakan langkah pra-pemrosesan yang wajib untuk algoritma berbasis jarak seperti KNN.

### Saran
1. **Penanganan Imbalanced Dataset**: Untuk eksperimen selanjutnya, disarankan menggunakan teknik resampling seperti **SMOTE** (Synthetic Minority Over-sampling Technique) agar model dapat mengenali kelas minoritas (kualitas 3, 4, dan 8) secara lebih akurat.
2. **Pengembangan Model Lain**: Dapat dicoba algoritma klasifikasi lain seperti *Random Forest*, *Support Vector Machine (SVM)*, atau *XGBoost* untuk membandingkan performanya dengan KNN.